# Project data exploration and scatter plots - looking at relplot

**Group name:** NBA SHOT ANALYSIS AND PREDICTIONS
**Created by:** Anthony Moll, Zhanxiang He, Kevin Medina
**Last updated:** 10/21/2026

**Dataset:** NBA Shot Logs from the 2014 to 2015 season (Kaggle). Each row is one shot.

### Import your data

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

df = pd.read_csv("shot_logs.csv")
print(df.shape)
df.head()

## Q1: Data background

The data is the NBA Shot Logs dataset from Kaggle. It has shots taken during the 2014 to 2015 NBA season that were recorded by player tracking cameras. There are 128,069 rows and 21 columns, and each row is one shot. For every shot it lists the shooter, the closest defender, how far the shot was, how far away the defender was, the shot clock, the number of dribbles, how long the shooter held the ball, the period of the game, and whether the shot went in.

We picked this dataset because we want to understand what makes a shot more likely to go in. Later on we want to use the information we have before the shooter lets go of the ball to predict if a shot will be made or missed.

## Q2: Show the number of missing data in each column of the dataframe

In [ ]:
missing = df.isna().sum()
missing_df = pd.DataFrame({"missing": missing, "percent": (missing / len(df) * 100).round(2)})
missing_df.sort_values("missing", ascending=False)

Only one column has missing values, which is SHOT_CLOCK. It is missing in 5,567 rows, about 4.35% of the data, and every other column is complete. Some of these look like shots at the very end of a period, since the third row has 0:00 on the game clock, so the shot clock may have been turned off. This should not change much for our plots since none of them use SHOT_CLOCK and it is a small part of the data. If we use shot clock when we train models we will need to drop those rows or fill them in.

## Q3: Show the data type for each column of the dataframe

In [ ]:
print("Negative TOUCH_TIME rows:", (df["TOUCH_TIME"] < 0).sum())
df.dtypes

Most of the types make sense. Measurements like SHOT_DIST, CLOSE_DEF_DIST, TOUCH_TIME and SHOT_CLOCK are floats, and counts like DRIBBLES, PERIOD and PTS_TYPE are ints. A few are not what I expected. GAME_CLOCK is stored as text like "1:09" instead of a number, so we would have to change it to seconds to use it. MATCHUP is also text with the date and both teams in one string. LOCATION, W and SHOT_RESULT are text labels, which is fine for categories.

Most columns are clear from their names, but some need the Kaggle description, like FGM (1 for made and 0 for missed), PTS_TYPE (2 or 3) and W (win or loss for the game). TOUCH_TIME also has some negative values, which should not be possible. There are 312 of them and we plan to remove those rows before training models.

## Q4: Your own question using relplot()
**Question:** How does field goal percentage change as shot distance gets bigger, and is it different for 2 point and 3 point shots?

In [ ]:
df["DIST_FT"] = df["SHOT_DIST"].round().astype(int)

g = sns.relplot(
    data=df[df["DIST_FT"] <= 35],
    x="DIST_FT", y="FGM", hue="PTS_TYPE", kind="line",
    errorbar=None, height=5, aspect=1.5, marker="o", palette="deep",
)
g.set_axis_labels("Shot distance (feet)", "Fraction of shots made")
g.figure.suptitle("Field goal percentage by shot distance", y=1.03)
plt.show()

For 2 point shots the percentage is highest near the basket, about 65% under 4 feet. It drops to about 40% by 10 feet, stays close to that until around 20 feet, and then slowly goes down. 3 point shots mostly show up from about 21 feet on, start around 30 to 40%, and fall to about 10% near 30 feet. Right around the 3 point line, 3 point shots went in a little more often than 2 point shots from the same distance, which I did not expect. Overall it is mostly what I thought would happen, since shots that are closer are easier.

A few things look strange. There are 3 point shots listed at short distances with 0%, which is probably bad data since nobody shoots a 3 from 5 feet. The lines also jump around past about 27 feet, probably because there are very few shots that far out.

## Q5: Your own question using catplot()
**Question:** Are players more accurate on catch and shoot shots (no dribbles) than after dribbling?

In [ ]:
df["DRIBBLE_GROUP"] = pd.cut(
    df["DRIBBLES"], bins=[-1, 0, 2, 6, 100], labels=["0", "1 to 2", "3 to 6", "7 or more"]
)

g = sns.catplot(
    data=df, x="DRIBBLE_GROUP", y="FGM", hue="PTS_TYPE", kind="bar",
    errorbar=None, height=5, aspect=1.4, palette="deep",
)
g.set_axis_labels("Dribbles before the shot", "Fraction of shots made")
g.figure.suptitle("Field goal percentage by number of dribbles", y=1.03)
plt.show()

Yes, shots with no dribbles were the most accurate for both shot types. 2 point shots went in about 55% of the time with 0 dribbles compared to about 43 to 46% after dribbling, and 3 point shots went in about 37% of the time with 0 dribbles compared to about 30 to 34% after dribbling. After the first dribble the percentages stay pretty close to each other, and the 3 point bar for 7 or more dribbles is a little higher than the middle groups, which I did not expect. Dribbling might not be the only reason for the drop, since shots after dribbling could also be harder in other ways, like having a defender closer.

## Q6: Create a plot with small multiples plots

In [ ]:
sample = df.sample(6000, random_state=42)

g = sns.relplot(
    data=sample, x="SHOT_DIST", y="CLOSE_DEF_DIST",
    col="PERIOD", col_wrap=3, height=5, s=20,
)
g.set_axis_labels("Shot distance (feet)", "Closest defender distance (feet)")
g.set_titles("Period {col_name}")
g.figure.suptitle("Shot distance vs. closest defender distance in each period (periods 5 to 7 are overtime)", y=1.02)
plt.show()

Periods 1 to 4 look very similar to each other. Most shots are close to the basket, under 10 feet, with the closest defender usually within about 5 feet. As the shot distance gets to around 20 to 25 feet the points spread out more, and the closest defender is sometimes 10 to 20 feet away, which are the more open 3 point shots. I do not see a clear difference between the four regular periods. The overtime periods (5, 6 and 7) only have a few points, so it is hard to say anything about them.